# Train the password strength models in Kaggle

This self-contained notebook reads four Kaggle datasets attached through **Add Data**; it does not clone GitHub. The classifier is our proposed extension to the password-manager paper, which lists ML risk analysis as future work but does not provide ML accuracy results.

1. [Password Strength Classifier](https://www.kaggle.com/datasets/bhavikbb/password-strength-classifier-dataset)
2. [RockYou common password list](https://www.kaggle.com/datasets/wjburns/common-password-list-rockyoutxt)
3. [10,000 most common passwords](https://www.kaggle.com/datasets/shivamb/10000-most-common-passwords)
4. [Top 10 Million Passwords](https://www.kaggle.com/datasets/joebeachcapital/top-10-million-passwords) (SecLists list, not the Xato dataset)

The notebook holds out 20% of unique weak-list examples before training and dictionary construction. It caps each class at 100,000 rows and balances weak and strong examples across their available sources to reduce source shortcuts. It records source counts. Strong training examples are randomly generated, so runs can differ. Keep the notebook private; do not publish raw password data or the dictionary.


In [ ]:
%pip install -q pandas numpy scikit-learn xgboost joblib zxcvbn-python
from pathlib import Path
import csv, json, math, os, random, re, secrets, string, zipfile
from collections import Counter
from importlib.metadata import version
import numpy as np
import pandas as pd
from IPython.display import display, FileLink
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from zxcvbn import zxcvbn
import joblib
SEED = 42
random.seed(SEED); np.random.seed(SEED)
WORK = Path('/kaggle/working/password_manager_training')
CHECKPOINTS = WORK / 'checkpoints'; CHECKPOINTS.mkdir(parents=True, exist_ok=True)
def checkpoint(name, payload):
    joblib.dump(payload, CHECKPOINTS / f'{name}.joblib')
print('Working directory:', WORK)

In [ ]:
# Find the attached files by their content/name; Kaggle changes input folder slugs sometimes.
all_files = [p for p in Path('/kaggle/input').rglob('*') if p.is_file()]
print('Attached files:')
for p in all_files: print(p)
csv_files = [p for p in all_files if p.suffix.lower() == '.csv']
def csv_columns(path):
    try:
        return {str(c).strip().lower() for c in pd.read_csv(path, nrows=0).columns}
    except Exception: return set()
labeled_candidates = [p for p in csv_files if {'password','strength'} <= csv_columns(p)]
rockyou_candidates = [p for p in all_files if 'rockyou' in p.name.lower() and p.suffix.lower() != '.csv']
common_candidates = [p for p in csv_files if 'common' in str(p).lower() or '10000' in str(p).lower()]
million_candidates = [p for p in all_files if ('million' in str(p).lower() or 'top-10' in str(p).lower()) and p not in rockyou_candidates]
assert labeled_candidates, 'Could not find a CSV with password and strength columns. Attach the Password Strength Classifier dataset.'
assert rockyou_candidates, 'Could not find rockyou.txt. Attach the RockYou dataset.'
assert common_candidates, 'Could not find the 10k common-password CSV. Attach the 10,000 passwords dataset.'
assert million_candidates, 'Could not find the Top 10 Million dataset. Attach it in Kaggle Add Data.'
LABELED, ROCKYOU, COMMON, MILLION = labeled_candidates[0], rockyou_candidates[0], common_candidates[0], million_candidates[0]
print('Using:', LABELED, ROCKYOU, COMMON, MILLION, sep='\n')
checkpoint('01_input_inventory', {'files':[str(p) for p in all_files], 'selected':[str(LABELED),str(ROCKYOU),str(COMMON),str(MILLION)]})

## Prepare source-separated training and evaluation data
Labels are 0 = weak, 1 = medium, and 2 = strong. The three common/leaked-password lists are treated as weak examples; the Password Strength Classifier labels supply medium and strong examples. A held-out part of the common-password lists is reserved for testing and excluded from both training and the dictionary. These are source-derived labels, not measured crackability ground truth. The paper you provided describes a password manager and lists ML risk analysis as future work; it does not publish classifier accuracy results.


In [ ]:
def normalize_label(v):
    try:
        n=int(v); return n if n in (0,1,2) else None
    except Exception: pass
    s=str(v).strip().lower()
    return {'weak':0,'low':0,'medium':1,'moderate':1,'strong':2,'high':2}.get(s)
labeled = pd.read_csv(LABELED, usecols=lambda c: str(c).strip().lower() in ('password','strength'))
labeled.columns = [str(c).strip().lower() for c in labeled.columns]
labeled = labeled[['password','strength']].dropna()
labeled['label'] = labeled['strength'].map(normalize_label)
labeled = labeled.dropna(subset=['label'])[['password','label']]
labeled['password'] = labeled.password.astype(str).str.strip()
labeled=labeled[labeled.password.str.len().between(1,128)].drop_duplicates('password').reset_index(drop=True)
labeled['_key']=labeled.password.str.lower()
labeled_train,labeled_test=train_test_split(labeled,test_size=0.20,random_state=SEED,stratify=labeled.label)
def read_password_csv(path):
    frame=pd.read_csv(path,dtype=str,on_bad_lines='skip')
    cols={str(c).strip().lower():c for c in frame.columns}
    col=cols.get('password',frame.columns[0])
    return frame[col].dropna().astype(str).str.strip()
common_passwords=read_password_csv(COMMON)
million_frame=pd.read_csv(MILLION,dtype=str,nrows=500000,on_bad_lines='skip')
million_col={str(c).strip().lower():c for c in million_frame.columns}.get('password',million_frame.columns[-1])
million_passwords=million_frame[million_col].dropna().astype(str).str.strip()
with open(ROCKYOU,encoding='utf-8',errors='ignore') as f:
    rockyou=pd.Series((line.rstrip('\r\n') for _,line in zip(range(200000),f)),dtype='string')
weak_all=pd.concat([common_passwords,million_passwords,rockyou],ignore_index=True).dropna().astype(str).str.strip()
weak_all=weak_all[weak_all.str.len().between(3,128)]
weak_pool=pd.DataFrame({'password':weak_all}).drop_duplicates('password').reset_index(drop=True)
weak_pool['_key']=weak_pool.password.str.lower()
weak_pool=weak_pool.drop_duplicates('_key').reset_index(drop=True)
weak_train,weak_test=train_test_split(weak_pool,test_size=0.20,random_state=SEED)
weak_train=weak_train.copy(); weak_test=weak_test.copy()
weak_train['label']=0; weak_test['label']=0
def random_strong():
    alphabet=string.ascii_letters+string.digits+string.punctuation
    n=random.randint(14,24)
    return ''.join(secrets.choice(alphabet) for _ in range(n))
strong_df=pd.DataFrame({'password':[random_strong() for _ in range(50000)],'label':2})
# Known weak-list entries take the weak label; exclude them from classifier-labeled rows to avoid contradictory labels.
weak_test_keys=set(weak_test['_key'])
all_weak_keys=set(weak_pool['_key'])
labeled_train=labeled_train[~labeled_train['_key'].isin(all_weak_keys)].copy()
# Keep the training signal mixed by source: weak examples include both labeled
# non-dictionary passwords and common-list passwords; strong includes real labeled
# examples and generated examples. This prevents source/dictionary shortcuts.
weak_label_pool=labeled_train[labeled_train.label==0][['password','label']].copy()
medium_pool=labeled_train[labeled_train.label==1][['password','label']].copy()
strong_labeled_pool=labeled_train[labeled_train.label==2][['password','label']].copy()
weak_list_pool=weak_train[['password','label']].copy()
strong_synthetic_pool=strong_df[['password','label']].copy()
per_class=min(100000,len(medium_pool),len(weak_label_pool)+len(weak_list_pool),len(strong_labeled_pool)+len(strong_synthetic_pool))
assert per_class>0,'One or more training classes are empty after source filtering.'
def sample_n(frame,n):
    return frame.sample(n=n,random_state=SEED) if n else frame.iloc[:0].copy()
# Reserve half of weak and strong examples for each source when available.
weak_label_n=min(len(weak_label_pool),(per_class+1)//2)
weak_list_n=min(len(weak_list_pool),per_class-weak_label_n)
weak_label_n=min(len(weak_label_pool),per_class-weak_list_n)
weak_list_n+=min(len(weak_list_pool)-weak_list_n,per_class-weak_label_n-weak_list_n)
weak_label_n+=min(len(weak_label_pool)-weak_label_n,per_class-weak_label_n-weak_list_n)
strong_labeled_n=min(len(strong_labeled_pool),per_class//2)
strong_synthetic_n=min(len(strong_synthetic_pool),per_class-strong_labeled_n)
strong_labeled_n=min(len(strong_labeled_pool),per_class-strong_synthetic_n)
strong_synthetic_n+=min(len(strong_synthetic_pool)-strong_synthetic_n,per_class-strong_labeled_n-strong_synthetic_n)
strong_labeled_n+=min(len(strong_labeled_pool)-strong_labeled_n,per_class-strong_labeled_n-strong_synthetic_n)
train_df=pd.concat([sample_n(weak_label_pool,weak_label_n),sample_n(weak_list_pool,weak_list_n),sample_n(medium_pool,per_class),sample_n(strong_labeled_pool,strong_labeled_n),sample_n(strong_synthetic_pool,strong_synthetic_n)],ignore_index=True)
assert train_df.label.value_counts().nunique()==1,'Training classes are not balanced.'
# Dictionary comes only from the weak-list training partition, never labeled or held-out rows.
dictionary=set(weak_train.password.astype(str).str.lower())
print('Training source mix:',{'weak_labeled':weak_label_n,'weak_common_lists':weak_list_n,'medium_labeled':per_class,'strong_labeled':strong_labeled_n,'strong_generated':strong_synthetic_n})
def dictionary_match(password):
    lowered=str(password).lower()
    return lowered in dictionary or bool(re.sub(r'\d+$','',lowered) in dictionary)
# Test weak generalization on held-out lists; test medium/strong on labeled holdout rows.
weak_test_candidates=weak_test[~weak_test.password.map(dictionary_match)].copy()
weak_test_candidates=weak_test_candidates[['password','label']]
weak_test_candidates['source']='heldout_common_lists'
train_keys=set(train_df.password.astype(str).str.lower())
heldout=labeled_test.copy()
heldout=heldout[~heldout['_key'].isin(train_keys)]
heldout=heldout[~heldout['_key'].isin(all_weak_keys)]
heldout=heldout[~heldout.password.map(dictionary_match)]
heldout=heldout[~heldout['_key'].isin(weak_test_keys)]
heldout=heldout[heldout.label.isin([1,2])][['password','label']].copy()
heldout['source']='heldout_labeled_csv'
test_candidates=pd.concat([weak_test_candidates,heldout],ignore_index=True)
test_parts=[]
for cls,group in test_candidates.groupby('label'):
    if len(group): test_parts.append(group.sample(n=min(5000,len(group)),random_state=SEED))
test_df=pd.concat(test_parts,ignore_index=True) if test_parts else pd.DataFrame(columns=['password','label','source'])
assert set(test_df.label.astype(int).unique())=={0,1,2},'Insufficient held-out examples in one or more classes.'
source_counts={'labeled_csv_rows':len(labeled),'labeled_csv_train_pool':len(labeled_train),'labeled_csv_holdout_pool':len(labeled_test),'common_passwords_csv_rows':len(common_passwords),'top_million_csv_rows_used':len(million_passwords),'rockyou_first_200k_rows':len(rockyou),'unique_weak_list_passwords':len(weak_pool),'weak_list_train_rows':len(weak_train),'weak_list_heldout_rows':len(weak_test),'synthetic_strong_generated':len(strong_df),'balanced_train_rows':len(train_df),'dictionary_entries':len(dictionary),'evaluation_rows':len(test_df)}
print('Rows by source:',source_counts)
print('Training rows:',len(train_df),' Evaluation rows:',len(test_df))
print('Training classes:',train_df.label.value_counts().sort_index().to_dict())
print('Evaluation classes:',test_df.label.value_counts().sort_index().to_dict())
print('Evaluation sources:',test_df.groupby(['label','source']).size().to_dict())
checkpoint('02_data_split',{'seed':SEED,'train_rows':len(train_df),'test_rows':len(test_df),'train_class_counts':train_df.label.value_counts().to_dict(),'test_class_counts':test_df.label.value_counts().to_dict(),'source_counts':source_counts,'holdout_protocol':'weak list holdout plus labeled CSV medium/strong; held-out weak list excluded from training and dictionary'})


## Extract the 17 model features and train both models

The dictionary membership feature is part of the proposed model. The exact training dictionary is included in the private model ZIP so the app can use the same feature values.

In [ ]:
FEATURE_NAMES=['length','entropy','count_uppercase','ratio_uppercase','count_lowercase','ratio_lowercase','count_digits','ratio_digits','count_symbols','ratio_symbols','count_unique_chars','ratio_unique_chars','sequences','repeats','transitions','dictionary_hit','common_patterns']
print('Independent weak-list training dictionary entries:',len(dictionary))
def features(password):
    p=str(password); n=len(p); counts=Counter(p); ent=-sum((v/n)*math.log2(v/n) for v in counts.values()) if n else 0.0
    up=sum(c.isupper() for c in p); low=sum(c.islower() for c in p); dig=sum(c.isdigit() for c in p); sym=sum(not c.isalnum() for c in p); uniq=len(set(p))
    pl=p.lower(); seq=sum(1 for i in range(max(0,n-2)) if ord(pl[i+2])-ord(pl[i+1])==1 and ord(pl[i+1])-ord(pl[i])==1)
    seq += sum(row[i:i+3] in pl for row in ('qwertyuiop','asdfghjkl','zxcvbnm') for i in range(len(row)-2))
    transitions=sum((0 if a.islower() else 1 if a.isupper() else 2 if a.isdigit() else 3)!=(0 if b.islower() else 1 if b.isupper() else 2 if b.isdigit() else 3) for a,b in zip(p,p[1:]))
    clean=re.sub(r'\d+$','',pl)
    values=[n,ent,up,up/n if n else 0,low,low/n if n else 0,dig,dig/n if n else 0,sym,sym/n if n else 0,uniq,uniq/n if n else 0,seq,sum(a==b for a,b in zip(p,p[1:])),transitions,int(pl in dictionary or bool(clean and clean in dictionary)),int(bool(re.search(r'(19|20)\d{2}|\d{4,8}',p)))]
    return values
X_train=np.asarray([features(p) for p in train_df.password], dtype=np.float32); y_train=train_df.label.to_numpy(dtype=int)
X_test=np.asarray([features(p) for p in test_df.password], dtype=np.float32); y_test=test_df.label.to_numpy(dtype=int)
feature_names=list(FEATURE_NAMES); joblib.dump(feature_names, WORK/'feature_names.joblib')
rf=RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, class_weight='balanced_subsample')
rf.fit(X_train,y_train); joblib.dump(rf, CHECKPOINTS/'random_forest.joblib'); print('Random Forest checkpoint saved')
xgb=XGBClassifier(n_estimators=350,max_depth=7,learning_rate=0.08,subsample=0.9,colsample_bytree=0.9,objective='multi:softprob',num_class=3,eval_metric='mlogloss',random_state=SEED,n_jobs=-1)
xgb.fit(X_train,y_train); joblib.dump(xgb,CHECKPOINTS/'xgboost.joblib'); joblib.dump(feature_names,CHECKPOINTS/'feature_names.joblib'); print('XGBoost checkpoint saved')

## Evaluate against the app's existing meter
The attached paper does not report classifier accuracy or macro-F1, so there are no base-paper ML scores to copy into this table. This run compares the existing app rule meter and zxcvbn with the two trained models on the source-separated evaluation set. Report these results as this project's experiment, not as a reproduction of paper results.


In [ ]:
def rule_predict(password):
    # Match existing-approach/client/src/lib/strengthChecker.ts, then group its 5 levels into 3.
    p=str(password)
    if not p: return 0
    raw=int(len(p)>=12)+int(bool(re.search(r'[A-Z]',p)))+int(bool(re.search(r'[a-z]',p)))+int(bool(re.search(r'[0-9]',p)))+int(bool(re.search(r'[^A-Za-z0-9]',p)))
    score=min(4,math.floor(raw*4/5))
    return 0 if score<=1 else (1 if score==2 else 2)
zxcvbn_scores=np.asarray([zxcvbn(str(p))['score'] for p in test_df.password],dtype=int)
print('zxcvbn raw score counts (0-4):',pd.Series(zxcvbn_scores).value_counts().sort_index().to_dict())
print('zxcvbn-to-class mapping: scores 0-1 -> weak, 2-3 -> medium, 4 -> strong')
zxcvbn_classes=np.where(zxcvbn_scores<=1,0,np.where(zxcvbn_scores<=3,1,2))
predictions={'Existing app rule meter':np.asarray([rule_predict(p) for p in test_df.password]),'zxcvbn':zxcvbn_classes,'Random Forest':rf.predict(X_test),'XGBoost':xgb.predict(X_test)}
current=[]
for name,pred in predictions.items(): current.append({'Model':name,'Accuracy (%)':100*accuracy_score(y_test,pred),'Macro F1 (%)':100*f1_score(y_test,pred,average='macro')})
comparison=pd.DataFrame(current)
display(comparison.round(2))
for name,pred in predictions.items():
    print('\n',name,'per-class report')
    print(classification_report(y_test,pred,labels=[0,1,2],target_names=['weak','medium','strong'],zero_division=0))
comparison.to_csv(WORK/'model_comparison.csv',index=False)
checkpoint('03_evaluation', {'current':current,'test_rows':len(test_df),'test_sources':test_df.groupby(['label','source']).size().to_dict(),'seed':SEED,'feature_names':feature_names})


## Package both trained models for the app
The ZIP includes Random Forest and XGBoost (saved in portable JSON format), the 17-feature order, the exact training dictionary, metadata, and fresh evaluation results. The API selects the packaged model with the highest test macro-F1. It excludes the full attached password lists. Keep the ZIP private because the dictionary is derived from common and leaked-password sources.


In [ ]:
joblib.dump(rf, WORK/'rf_model.joblib')
xgb.save_model(WORK/'xgb_model.json')
joblib.dump(feature_names, WORK/'feature_names.joblib')
model_rows=comparison[comparison['Model'].isin(['Random Forest','XGBoost'])]
selected_row=model_rows.sort_values('Macro F1 (%)',ascending=False).iloc[0]
selected_model='rf' if selected_row['Model']=='Random Forest' else 'xgb'
metadata={'seed':SEED,'available_models':['rf','xgb'],'selected_model':selected_model,'model_metrics':model_rows.to_dict(orient='records'),'features':feature_names,'class_labels':{'0':'weak','1':'medium','2':'strong'},'train_rows':len(train_df),'test_rows':len(test_df),'dataset_sources':[str(LABELED),str(ROCKYOU),str(COMMON),str(MILLION)],'source_counts':source_counts,'dictionary_entries':len(dictionary),'evaluation_protocol':'held-out weak-list partition plus held-out labeled medium/strong examples','base_paper':'Password Manager, IJRASET 2025, DOI 10.22214/ijraset.2025.76363; it reports no ML classifier accuracy metrics','package_versions':{name:version(name) for name in ('pandas','numpy','scikit-learn','xgboost','joblib','zxcvbn-python')},'note':'Top 10 Million Kaggle source is a SecLists common-password list, not the Xato dataset. The base paper reports no ML accuracy scores. Current metrics use source-derived labels and are not real-world crackability measurements.'}
(WORK/'training_metadata.json').write_text(json.dumps(metadata,indent=2))
(WORK/'dictionary.txt').write_text('\n'.join(sorted(dictionary))+'\n',encoding='utf-8')
archive=Path('/kaggle/working/password-manager-kaggle-model.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ('rf_model.joblib','xgb_model.json','feature_names.joblib','training_metadata.json','model_comparison.csv','dictionary.txt'):
        z.write(WORK/name,arcname=name)
checkpoint('04_packaging', {'archive':str(archive),'included':['rf_model.joblib','xgb_model.json','feature_names.joblib','training_metadata.json','model_comparison.csv','dictionary.txt'],'selected_model':selected_model})
print('Download and attach this file to Codex:')
display(FileLink(str(archive)))